In [ ]:
!pip install rapidfuzz

In [ ]:
import requests
from google.colab import userdata
from rapidfuzz import fuzz
import json
import random
API_KEY = userdata.get('PoTKey')


In [ ]:
def prompt(message, model="openai/gpt-4.1-mini", max_tokens=100):
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": model,
            "messages": [{"role": "user", "content": message}],
            "max_tokens": max_tokens
        }
    )
    if response.status_code != 200:
        print(f"API Error: {response.status_code} - {response.text}")
        return "" # Return empty string or raise an exception
    data = response.json()
    if 'choices' not in data or not data['choices']:
        print(f"Unexpected API response format: {data}")
        return "" # Return empty string or raise an exception
    message_obj = data['choices'][0]['message']
    content = message_obj.get('content') or message_obj.get('reasoning', '')
    return content.strip()
from rapidfuzz import fuzz

def removeTarget(varList, target, threshold):
    scores = {x: fuzz.ratio(x.lower(), target.lower()) for x in varList}
    bestMatch = max(scores, key=scores.get)
    if scores[bestMatch] >=threshold:
        varList.remove(bestMatch)
    return varList

In [ ]:
PROMPT_TEMPLATES = {
    1: """Extract causal variables from this scenario as a comma-separated list of events.

Scenario: {context}

Rules:
- Extract only events that directly cause or are caused by other events
- Maximum 5 variables

Output ONLY the comma-separated list:""",

    2: """Extract up to 5 causal variables from this scenario.
A causal variable is an event, action, condition, or outcome that directly causes or is caused by another event.

Rules:
- Extract only causally connected variables
- Do not extract entities, descriptions, background facts, or locations
- Do not extract intentions, goals, or plans
- Do not extract duplicate or overlapping variables
- Every variable must make sense as a standalone phrase

Output ONLY a comma-separated list.

Scenario: {context}""",

    3: """Extract causal variables from this scenario as a comma-separated list.

A causal variable is a self-contained event, action, condition, or outcome that directly causes or is directly caused by another variable in the scenario.

EXAMPLES:

Scenario: A bridge in Baltimore collapsed after a ship collided with one of its support columns, leading to a state of emergency being declared and search and rescue efforts for survivors. The ship had lost power before the collision.
Output: ship losing power, ship colliding with bridge support column, bridge collapse, state of emergency declared, search and rescue efforts launched

Scenario: Marie Kondo shifted her mindset about mess and perfection after having three children. She now values spending time with her kids over maintaining perfect order in her home.
Output: having three children, Marie Kondo's mindset shifting, spending time with kids prioritized, maintaining perfect order deprioritized

Scenario: A study revealed that a sudden rise in the number of silver coins in circulation in medieval England was due to melted down Byzantine treasures. This practice kickstarted the economy of England and established a monetary system that lasted for a millennium.
Output: melting down Byzantine treasures, rise in silver coins in circulation, economy of England kickstarted, monetary system established

Scenario: Nicole Flattery shares her favorite swimming spot and her admiration for various films and TV shows. She also discusses her preference for using a dumb phone.
Output:

Scenario: Jair Bolsonaro spent two nights hiding in the Hungarian embassy as federal police closed in on his allies amid ongoing criminal investigations.
Output: federal police closing in on allies, criminal investigations into Bolsonaro, Bolsonaro hiding in Hungarian embassy

Now extract from this scenario. Output ONLY the comma-separated list, no explanation:

Scenario: {context}""",
    4: """Extract up to 5 causal variables from this scenario.

A causal variable is a self-contained event, action, condition, or outcome that participates in a direct cause→effect relationship within the scenario.

Rules:

* Extract only variables that directly cause or are directly caused by another variable in the scenario
* Do not extract entities, descriptions, background facts, locations, or time periods
* Do not extract intentions, goals, plans, opinions, or hypothetical outcomes
* Do not extract duplicate, overlapping, or narrowly reworded versions of the same causal event
* Merge closely related events into a single broader causal variable when appropriate
* Prefer broader causal nodes over narrow details
* Every variable must make sense as a standalone phrase
* Variables should describe something happening, changing, increasing, decreasing, causing, preventing, enabling, or resulting

Output ONLY a comma-separated list.

Scenario: {context}
""",
    5:"""Extract up to 5 causal variables from this scenario.

A causal variable is a self-contained event, action, condition, or outcome that directly causes or is directly caused by another variable in the scenario.

Only extract variables that are part of an explicit or strongly implied causal chain.

Rules:

* Extract only variables that participate in a cause→effect relationship within the scenario
* Do not extract entities, descriptions, background facts, locations, or time periods
* Do not extract intentions, goals, plans, opinions, or hypothetical outcomes
* Do not extract descriptive features, attributes, or static properties
* Do not extract duplicate, overlapping, or narrowly reworded versions of the same causal event
* Merge closely related events into a broader causal variable when appropriate
* Prefer broader causal nodes over narrow details
* Every variable must make sense as a standalone phrase
* Variables should describe something happening, changing, increasing, decreasing, causing, preventing, enabling, threatening, or resulting
* Do not extract themes, topics, discussions, or artistic content unless they directly participate in a causal relationship


Output ONLY a list seperated by the '|' symbol. Do not use any other divider such as a comma or arrow.

Scenario: {context}
""",
    6: """Extract the causal variables from this scenario.
    Scenario: {context}

    Only extract CAUSAL variables that either may cause an event or may be the result of an event.
    Output your answer as a list seperated by the '|' symbol. Do not use any other divider such as a comma or arrow.
    """
}

def returnVarList(context, prompt_type=5):
    template = PROMPT_TEMPLATES[prompt_type]
    message = template.format(context=context)
    raw = prompt(message, max_tokens=300)
    return raw

In [ ]:
import json
from tqdm import tqdm

with open('/content/CausalProbeHard-precomputed-v2.json', 'r') as f:
    causalDataPrecompute = json.load(f)

outputList = []

for item in tqdm(causalDataPrecompute, desc="Recomputing varList"):
    try:
        varList = returnVarList(item['context'], prompt_type=5)
        newEntry = {**item, 'varList': varList}
        outputList.append(newEntry)
    except Exception as e:
        print(f"Error on id {item['id']}: {e}")
        outputList.append(item)

with open('CausalProbeHard-precomputed-v2.json', 'w') as f:
    json.dump(outputList, f, indent=2)

print(f"\nDone. Saved {len(outputList)}/{len(causalDataPrecompute)} entries.")

Recomputing varList:   0%|          | 5/3461 [00:05<1:02:13,  1.08s/it]


KeyboardInterrupt: 

In [ ]:
for entry in causalDataPrecompute:
    if isinstance(entry['varList'], str):
        entry['varList'] = [v.strip() for v in entry['varList'].split('|') if v.strip() and len(v.strip()) > 3]

# save fixed version
with open('CausalProbeHard-precomputed-v2-fixed.json', 'w') as f:
    json.dump(causalDataPrecompute, f, indent=2)

print(f"Done. Sample varList: {causalDataPrecompute[0]['varList']}")

Done. Sample varList: ['Surge in interest in AI', 'Increased media coverage of AI', 'Journalists contributing to hype surrounding AI', 'Criticism of journalists for inaccurate AI reporting', 'Calls for more balanced AI reporting']


In [ ]:
with open('/content/CausalProbeHard-precomputed-v2-fixed.json', 'r') as f:
    testing_data = json.load(f)